# Transfermarkt: Enriquecimiento con valor de mercado

Este notebook incorpora el valor de mercado de los jugadores como atributo de la capa Gold, a partir de los datos públicos de Transfermarkt. Su motivación es operativa: el motor de similitud del notebook 05 propone jugadores del mismo perfil de estilo, pero sin una noción de coste recomendaría indistintamente a un futbolista de élite y a uno asequible para cubrir la misma necesidad. El valor de mercado es el filtro que convierte «jugadores del mismo perfil» en «jugadores del mismo perfil fichables», y es la pieza que completa la lógica de detección de oportunidades de mercado.

Coherente con la decisión metodológica del proyecto, el valor de mercado no interviene en el agrupamiento ni en el cálculo de similitud, es una variable de resultado, no de estilo, y agruparía a los jugadores por calidad esperada en lugar de por identidad de juego. Se incorpora exclusivamente como descriptor para el filtrado posterior, igual que la edad o la liga.

El reto técnico de esta sección no es el filtro, sino el emparejamiento entre dos fuentes independientes: los jugadores de Wyscout, identificados por su `player_id` en la capa Gold, y los registros de Transfermarkt, con su propio identificador. Ambas fuentes registran los nombres de forma heterogénea (distinto número de apellidos, transliteraciones, acentos, apodos), de modo que un cruce directo por nombre es inviable. La estrategia adoptada se apoya en un elemento discriminante común y fiable: la fecha de nacimiento. El emparejamiento procede por niveles, bloqueando siempre por fecha de nacimiento. Un emparejamiento determinista exacto sobre nombre normalizado y, para los casos no resueltos, una segunda pasada de similitud difusa de nombres restringida a quienes comparten fecha de nacimiento, lo que reduce los falsos positivos a un mínimo. Por último, la valoración se alinea temporalmente con la temporada 2017/18 analizada, tomando el valor contemporáneo a esa campaña y no el actual.

## Carga de fuentes

El enriquecimiento parte de dos orígenes. Del lado de Transfermarkt, dos ficheros del conjunto de datos público: el catálogo de jugadores —con nombre, fecha de nacimiento y posición — y el histórico de valoraciones de mercado, que asocia a cada jugador un valor en euros en distintas fechas. Del lado del proyecto, el universo de jugadores de la capa Gold, al que se añade desde la capa Bronze la fecha de nacimiento de Wyscout, que actuará como clave de emparejamiento.

Antes de diseñar el cruce conviene inspeccionar ambas fuentes: confirmar que la fecha de nacimiento está presente y sin apenas ausencias en los dos lados y verificar los formatos de nombre y fecha, que difieren entre orígenes y deberán normalizarse en la sección siguiente.

In [1]:
# Carga de fuentes: Transfermarkt (CSV) y jugadores Wyscout (Gold + Bronze)
import os, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

pd.set_option("display.max_columns", 50)
warnings.filterwarnings("ignore", category=UserWarning)

PROJECT_ROOT = Path.cwd().parents[0]
load_dotenv()
engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('PG_USER')}@{os.getenv('PG_HOST')}:{os.getenv('PG_PORT')}/{os.getenv('PG_DB')}"
)

TM_DIR = PROJECT_ROOT / "data" / "raw" / "transfermarkt"

tm_players = pd.read_csv(TM_DIR / "players.csv",
                         usecols=['player_id', 'name', 'date_of_birth',
                                  'country_of_birth', 'sub_position', 'position'])
tm_vals = pd.read_csv(TM_DIR / "player_valuations.csv",
                      usecols=['player_id', 'date', 'market_value_in_eur'])

wy_players = pd.read_sql("""
    SELECT g.player_id, g.short_name, g.position_group,
           b.first_name, b.last_name, b.complete_name, b.birth_date
    FROM gold.player_stats_per90 g
    LEFT JOIN bronze.wyscout_players b ON b.wy_player_id = g.player_id
""", engine)

print("FUENTES CARGADAS")
print(f"  Transfermarkt · players     : {len(tm_players):>7,}")
print(f"  Transfermarkt · valuations  : {len(tm_vals):>7,}")
print(f"  Wyscout · players (en Gold) : {len(wy_players):>7,}")
print(f"\nNULOS EN LA CLAVE DE EMPAREJAMIENTO (fecha de nacimiento)")
print(f"  Transfermarkt date_of_birth : {tm_players['date_of_birth'].isna().sum()}")
print(f"  Wyscout birth_date          : {wy_players['birth_date'].isna().sum()}")
print(f"\nMUESTRA — Wyscout")
print(wy_players[['short_name', 'complete_name', 'birth_date']].head(3).to_string(index=False))
print(f"\nMUESTRA — Transfermarkt")
print(tm_players[['name', 'date_of_birth', 'country_of_birth']].head(3).to_string(index=False))

FUENTES CARGADAS
  Transfermarkt · players     :  34,330
  Transfermarkt · valuations  : 447,281
  Wyscout · players (en Gold) :   2,561

NULOS EN LA CLAVE DE EMPAREJAMIENTO (fecha de nacimiento)
  Transfermarkt date_of_birth : 49
  Wyscout birth_date          : 0

MUESTRA — Wyscout
short_name complete_name birth_date
 D. Astori davide astori 1987-01-07
   L. Dunk    lewis dunk 1991-11-21
   M. Sarr   malang sarr 1999-01-23

MUESTRA — Transfermarkt
              name       date_of_birth country_of_birth
    Miroslav Klose 1978-06-09 00:00:00           Poland
Roman Weidenfeller 1980-08-06 00:00:00          Germany
  Dimitar Berbatov 1981-01-30 00:00:00         Bulgaria


## 1. Normalización de las claves de emparejamiento

El cruce se apoya en dos claves: la fecha de nacimiento, que actuará como bloqueo, y el nombre, que resolverá la coincidencia dentro de cada bloque. Ambas deben expresarse de forma idéntica en las dos fuentes antes de compararse.

La fecha de nacimiento se lleva a tipo fecha en los dos orígenes —en Transfermarkt requiere descartar la marca de tiempo que acompaña al valor—. El nombre se reduce a una forma canónica: sin acentos ni diacríticos, en minúsculas y sin signos de puntuación, de modo que «Müller», «Muller» y «müller» converjan a una misma cadena. El nombre de Wyscout ya se almacena en una forma próxima a la canónica, mientras que el de Transfermarkt conserva mayúsculas y formato original, por lo que la normalización iguala ambos a un mismo estándar. Sobre las cadenas normalizadas operará el emparejamiento de la sección siguiente.

In [2]:
# Normalización de fecha de nacimiento y nombre en ambas fuentes
import re
from unidecode import unidecode

def normalizar_nombre(s):
    if pd.isna(s):
        return None
    s = unidecode(str(s)).lower()
    s = re.sub(r"[^a-z\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

# Wyscout
wy = wy_players.copy()
wy['dob'] = pd.to_datetime(wy['birth_date']).dt.date
wy['name_norm'] = wy['complete_name'].apply(normalizar_nombre)
# respaldo: si complete_name viniera vacío, componer desde first/last
falta = wy['name_norm'].isna() | (wy['name_norm'] == "")
wy.loc[falta, 'name_norm'] = (wy.loc[falta, 'first_name'].fillna('') + ' ' +
                              wy.loc[falta, 'last_name'].fillna('')).apply(normalizar_nombre)

# Transfermarkt
tm = tm_players.copy()
tm['dob'] = pd.to_datetime(tm['date_of_birth'], errors='coerce').dt.date
tm['name_norm'] = tm['name'].apply(normalizar_nombre)
tm = tm.dropna(subset=['dob', 'name_norm'])

print("NORMALIZACIÓN COMPLETADA")
print(f"  Wyscout · name_norm vacíos : {(wy['name_norm'].fillna('') == '').sum()}")
print(f"  Wyscout · dob nulos        : {wy['dob'].isna().sum()}")
print(f"  TM tras descartar dob/nombre nulos: {len(tm):,} (de {len(tm_players):,})")
print(f"\nMUESTRA NORMALIZADA — Wyscout")
print(wy[['name_norm', 'dob']].head(4).to_string(index=False))
print(f"\nMUESTRA NORMALIZADA — Transfermarkt")
print(tm[['name_norm', 'dob']].head(4).to_string(index=False))

# Diagnóstico anticipado: ¿cuántas fechas de nacimiento Wyscout existen en TM?
dob_tm = set(tm['dob'])
cobertura_dob = wy['dob'].isin(dob_tm).mean() * 100
print(f"\nDIAGNÓSTICO — jugadores Wyscout cuya fecha de nacimiento existe en TM: {cobertura_dob:.1f}%")

NORMALIZACIÓN COMPLETADA
  Wyscout · name_norm vacíos : 0
  Wyscout · dob nulos        : 0
  TM tras descartar dob/nombre nulos: 34,281 (de 34,330)

MUESTRA NORMALIZADA — Wyscout
    name_norm        dob
davide astori 1987-01-07
   lewis dunk 1991-11-21
  malang sarr 1999-01-23
 ola toivonen 1986-07-03

MUESTRA NORMALIZADA — Transfermarkt
         name_norm        dob
    miroslav klose 1978-06-09
roman weidenfeller 1980-08-06
  dimitar berbatov 1981-01-30
             lucio 1978-05-08

DIAGNÓSTICO — jugadores Wyscout cuya fecha de nacimiento existe en TM: 99.8%


## 2. Emparejamiento por niveles

El cruce entre fuentes se resuelve en dos niveles, ambos bloqueados por fecha de nacimiento. La idea rectora es que coincidir en la fecha de nacimiento exacta —una de varios miles de combinaciones posibles— y además en el nombre es un suceso que difícilmente ocurre por azar entre dos jugadores distintos, de modo que el bloqueo por fecha reduce los falsos positivos a un mínimo.

El primer nivel es un emparejamiento determinista: se enlazan los jugadores que coinciden simultáneamente en nombre normalizado y fecha de nacimiento. Resuelve los casos en que el nombre se escribe de forma idéntica en ambas fuentes, que son la mayoría. El segundo nivel aborda los no resueltos —diferencias de transliteración, número de apellidos o uso de apodos— mediante similitud difusa de nombres, pero restringida a los candidatos de Transfermarkt que comparten exactamente la fecha de nacimiento del jugador de Wyscout. Para medir la similitud se emplea una métrica que es insensible al orden de las palabras y a la presencia de tokens adicionales, lo que resuelve los casos en que una fuente registra el nombre completo y la otra una forma abreviada. Solo se aceptan coincidencias por encima de un umbral de similitud exigente, y entre los candidatos de igual fecha se retiene el de mayor puntuación.

In [3]:
# Emparejamiento por niveles (exacto y difuso, bloqueados por fecha de nacimiento)
from rapidfuzz import fuzz, process

FUZZY_THRESHOLD = 85

# Nivel 1: match exacto (nombre normalizado + fecha de nacimiento)
tm_key = tm[['player_id', 'name_norm', 'dob']].rename(columns={'player_id': 'tm_id'})
n1 = wy.merge(tm_key, on=['name_norm', 'dob'], how='inner')
n1 = n1.drop_duplicates(subset='player_id', keep='first')
n1['match_level'] = 'exacto'
n1['match_score'] = 100.0

matched_ids = set(n1['player_id'])
pendientes = wy[~wy['player_id'].isin(matched_ids)].copy()

# Nivel 2: match difuso bloqueado por fecha de nacimiento
tm_por_dob = {d: g for d, g in tm_key.groupby('dob')}

filas_n2 = []
for _, r in pendientes.iterrows():
    cand = tm_por_dob.get(r['dob'])
    if cand is None or cand.empty:
        continue
    best = process.extractOne(
        r['name_norm'], cand['name_norm'].tolist(),
        scorer=fuzz.token_set_ratio, score_cutoff=FUZZY_THRESHOLD)
    if best is not None:
        nombre_tm, score, idx_local = best
        filas_n2.append({
            'player_id': r['player_id'], 'tm_id': cand.iloc[idx_local]['tm_id'],
            'match_level': 'difuso', 'match_score': float(score),
            'name_norm': r['name_norm'], 'name_norm_tm': nombre_tm,
        })

n2 = pd.DataFrame(filas_n2)

# Consolidación
cols = ['player_id', 'tm_id', 'match_level', 'match_score']
matches = pd.concat([n1[cols], n2[cols] if len(n2) else pd.DataFrame(columns=cols)],
                    ignore_index=True).drop_duplicates(subset='player_id', keep='first')

n_total = len(wy)
n_exacto = (matches['match_level'] == 'exacto').sum()
n_difuso = (matches['match_level'] == 'difuso').sum()
n_sin = n_total - len(matches)
print("EMPAREJAMIENTO POR NIVELES")
print(f"  Nivel 1 · exacto  : {n_exacto:>5}  ({100*n_exacto/n_total:.1f}%)")
print(f"  Nivel 2 · difuso  : {n_difuso:>5}  ({100*n_difuso/n_total:.1f}%)")
print(f"  Sin emparejar     : {n_sin:>5}  ({100*n_sin/n_total:.1f}%)")
print(f"  TOTAL emparejado  : {len(matches):>5}  ({100*len(matches)/n_total:.1f}%)")

if len(n2):
    print(f"\nMUESTRA DE MATCHES DIFUSOS (para auditar en §3):")
    print(n2[['name_norm', 'name_norm_tm', 'match_score']]
          .sort_values('match_score').head(12).to_string(index=False))

EMPAREJAMIENTO POR NIVELES
  Nivel 1 · exacto  :  1584  (61.9%)
  Nivel 2 · difuso  :   766  (29.9%)
  Sin emparejar     :   211  (8.2%)
  TOTAL emparejado  :  2350  (91.8%)

MUESTRA DE MATCHES DIFUSOS (para auditar en §3):
            name_norm      name_norm_tm  match_score
       thomas edwards       tom edwards    88.000000
         jose holebas     jose cholevas    88.000000
      oliver mcburnie      oli mcburnie    88.888889
      ibrahima m baye    ibrahima mbaye    89.655172
      abdallah n dour    abdallah ndour    89.655172
            tom smith       tommy smith    90.000000
          joshua sims         josh sims    90.000000
charly junior musonda charly musonda jr    90.322581
         ahmed benali      ahmad benali    91.666667
        sung yeung ki     sung yueng ki    92.307692
        siriki sanogo     siriky sanogo    92.307692
        dominik maroh     dominic maroh    92.307692


## 3. Auditoría de cobertura y casos no resueltos

El emparejamiento automático exige una doble verificación: que las coincidencias difusas aceptadas sean correctas —control de falsos positivos— y que los jugadores no emparejados lo estén por una razón legítima y no por un umbral mal calibrado —control de falsos negativos—.

Para lo primero se inspeccionan las coincidencias difusas de menor puntuación, que son las de mayor riesgo; su validez confirma que el umbral elegido no admite enlaces espurios. Para lo segundo se examinan los jugadores sin contraparte: dado que el universo es de tamaño abordable, la inspección de estos casos es viable y permite distinguir entre ausencias reales en la fuente —jugadores que Transfermarkt no recoge para esa fecha— y nombres tan divergentes que el umbral no alcanza a enlazar, susceptibles de recuperación. Los jugadores que finalmente queden sin valor de mercado se conservan en el sistema con ese campo vacío, de modo que no se filtran por precio pero permanecen disponibles para el resto de funciones.

In [4]:
# Inspección de los jugadores sin emparejar
sin_match = wy[~wy['player_id'].isin(set(matches['player_id']))].copy()

# ¿Su fecha de nacimiento existe en TM? Si sí, el fallo es de nombre (recuperable); si no, ausencia real
dob_tm = set(tm['dob'])
sin_match['dob_en_tm'] = sin_match['dob'].isin(dob_tm)

print(f"NO EMPAREJADOS: {len(sin_match)}")
print(f"  · con fecha de nacimiento presente en TM (fallo de nombre, recuperable): {sin_match['dob_en_tm'].sum()}")
print(f"  · sin fecha de nacimiento en TM (ausencia real en la fuente)          : {(~sin_match['dob_en_tm']).sum()}")

# Para los recuperables: ¿cuál sería su mejor candidato difuso por debajo del umbral?
tm_por_dob = {d: g for d, g in tm.groupby('dob')}
filas = []
for _, r in sin_match[sin_match['dob_en_tm']].iterrows():
    cand = tm_por_dob.get(r['dob'])
    best = process.extractOne(r['name_norm'], cand['name_norm'].tolist(),
                              scorer=fuzz.token_set_ratio)
    if best:
        filas.append({'wy_name': r['name_norm'], 'tm_best': best[0],
                      'score': round(best[1], 1), 'short_name': r['short_name']})

casi = pd.DataFrame(filas).sort_values('score', ascending=False)
print(f"\nMEJOR CANDIDATO (bajo umbral 85) DE LOS RECUPERABLES — top por score:")
print(casi.head(20).to_string(index=False))
print(f"\nDistribución de scores de los recuperables:")
print(casi['score'].describe().round(1).to_string())

NO EMPAREJADOS: 211
  · con fecha de nacimiento presente en TM (fallo de nombre, recuperable): 206
  · sin fecha de nacimiento en TM (ausencia real en la fuente)          : 5

MEJOR CANDIDATO (bajo umbral 85) DE LOS RECUPERABLES — top por score:
                   wy_name                tm_best  score      short_name
            shaquell moore             shaq moore   83.3        S. Moore
           daniel williams         danny williams   82.8     D. Williams
             bamidele alli              dele alli   81.8         D. Alli
        wilfride kanga aka         wilfried kanga   81.2        W. Kanga
  samuel castillejo azuaga        samu castillejo   80.0 Samu Castillejo
     eduardo exposito jaen           edu exposito   80.0    Edu Expósito
   papiss mison djilobodji        papy djilobodji   80.0   P. Djilobodji
     gianfilippo felicioli gian filippo felicioli   79.1    G. Felicioli
      gnaly maxwell cornet          maxwel cornet   78.8       M. Cornet
    manuel trigueros mun

### Recuperación de nombres legales frente a nombres usuales

La inspección de los no emparejados revela que solo cinco jugadores carecen realmente de contraparte en Transfermarkt; los restantes comparten fecha de nacimiento con algún registro y fallan únicamente por el nombre. Dentro de estos se distingue un patrón sistemático: Transfermarkt tiende a registrar el nombre usual o abreviado del jugador —«Dani Ceballos», «Cesc Fàbregas», «Dayot Upamecano»— mientras que Wyscout almacena el nombre legal completo —«Daniel Ceballos Fernández», «Francesc Fàbregas i Soler», «Dayotchanculle Upamecano»—. La métrica de similitud empleada en el segundo nivel penaliza los apellidos adicionales que una fuente incluye y la otra omite, dejando estas coincidencias —correctas— por debajo del umbral.

Para recuperarlas sin rebajar la exigencia global se aplica una tercera pasada, restringida de nuevo a quienes comparten fecha de nacimiento, con una métrica de similitud tolerante a los tokens omitidos: evalúa hasta qué punto el nombre más corto está contenido en el más largo. El bloqueo por fecha mantiene los falsos positivos en un mínimo, y un umbral elevado sobre esta métrica recupera los nombres legales frente a usuales sin admitir enlaces espurios. Las coincidencias recuperadas se auditan de nuevo antes de incorporarse.

In [5]:
# Tercera pasada: recuperación de diminutivos con métrica tolerante a tokens omitidos
RECUP_THRESHOLD = 88

# Reconstruye los grupos por fecha usando una columna 'tm_id' coherente
tm_dob = tm.rename(columns={'player_id': 'tm_id'})[['tm_id', 'name_norm', 'dob']]
tm_por_dob = {d: g.reset_index(drop=True) for d, g in tm_dob.groupby('dob')}

recuperables = sin_match[sin_match['dob_en_tm']].copy()
filas_n3 = []
for _, r in recuperables.iterrows():
    cand = tm_por_dob.get(r['dob'])
    if cand is None or cand.empty:
        continue
    best = process.extractOne(
        r['name_norm'], cand['name_norm'].tolist(),
        scorer=fuzz.partial_token_sort_ratio, score_cutoff=RECUP_THRESHOLD)
    if best is not None:
        nombre_tm, score, idx_local = best
        filas_n3.append({
            'player_id': r['player_id'], 'tm_id': cand.iloc[idx_local]['tm_id'],
            'match_level': 'difuso_recup', 'match_score': float(score),
            'wy_name': r['name_norm'], 'tm_name': nombre_tm,
        })

n3 = pd.DataFrame(filas_n3)
print(f"RECUPERADOS EN TERCERA PASADA: {len(n3)} de {len(recuperables)}")
if len(n3):
    print(f"\nAUDITORÍA — recuperados ordenados por score ascendente (revisar los más bajos):")
    print(n3[['wy_name', 'tm_name', 'match_score']]
          .sort_values('match_score').head(30).to_string(index=False))

RECUPERADOS EN TERCERA PASADA: 27 de 206

AUDITORÍA — recuperados ordenados por score ascendente (revisar los más bajos):
                          wy_name                tm_name  match_score
            daniel alves da silva             dani alves    88.888889
            gianfilippo felicioli gian filippo felicioli    89.473684
               wilfride kanga aka         wilfried kanga    92.857143
                   papakouli diop              pape diop    94.117647
         moussa sidi yaya dembele          mousa dembele    96.000000
        federico nicolas cartabia          fede cartabia   100.000000
                   shaquell moore             shaq moore   100.000000
            santiago mina lorenzo             santi mina   100.000000
         maximiliano gaston lopez             maxi lopez   100.000000
           edinaldo gomes pereira                  naldo   100.000000
              roberto suarez pier             rober pier   100.000000
      ronaldo aparecido rodrigues     

In [6]:
# Cuarta pasada: umbral relajado con auditoría previa (bloqueo por fecha intacto)
RECUP4_THRESHOLD = 80

# Quiénes siguen sin emparejar tras las tres pasadas anteriores
ya_emparejados = set(matches['player_id']) | (set(n3['player_id']) if len(n3) else set())
pendientes4 = wy[~wy['player_id'].isin(ya_emparejados)].copy()
pendientes4 = pendientes4[pendientes4['dob'].isin(dob_tm)]   # solo recuperables (fecha en TM)

filas_n4 = []
for _, r in pendientes4.iterrows():
    cand = tm_por_dob.get(r['dob'])          # tm_por_dob ya está construido en la 3ª pasada (tm_id, name_norm, dob)
    if cand is None or cand.empty:
        continue
    best = process.extractOne(
        r['name_norm'], cand['name_norm'].tolist(),
        scorer=fuzz.partial_token_sort_ratio, score_cutoff=RECUP4_THRESHOLD)
    if best is not None:
        nombre_tm, score, idx_local = best
        filas_n4.append({
            'player_id': r['player_id'], 'tm_id': cand.iloc[idx_local]['tm_id'],
            'match_level': 'difuso_recup2', 'match_score': float(score),
            'short_name': r['short_name'], 'wy_name': r['name_norm'], 'tm_name': nombre_tm,
        })

n4 = pd.DataFrame(filas_n4)
print(f"CANDIDATOS CUARTA PASADA (umbral {RECUP4_THRESHOLD}): {len(n4)} de {len(pendientes4)} recuperables pendientes\n")
if len(n4):
    print("AUDITORÍA — ordenados por score ASCENDENTE (los de arriba son los dudosos, revísalos uno a uno):")
    print(n4[['short_name', 'wy_name', 'tm_name', 'match_score']]
          .sort_values('match_score').to_string(index=False))

CANDIDATOS CUARTA PASADA (umbral 80): 36 de 179 recuperables pendientes

AUDITORÍA — ordenados por score ASCENDENTE (los de arriba son los dudosos, revísalos uno a uno):
      short_name                           wy_name          tm_name  match_score
    Paco Alcácer          francisco alcacer garcia     paco alcacer    80.000000
       D. Torres     daniel alejandro torres rojas      dani torres    80.000000
    D. Upamecano          dayotchanculle upamecano  dayot upamecano    80.000000
             Ivi                ivan lopez alvarez        ivi lopez    80.000000
    Dani Pacheco             daniel pacheco lobato     dani pacheco    80.000000
         Etxeita       xabier etxeita gorritxategi     xabi etxeita    80.000000
      Javi López            javier lopez rodriguez       javi lopez    80.000000
      Pepe Reina            jose manuel reina paez       pepe reina    80.000000
    W. Caballero         wilfredo daniel caballero  willy caballero    80.000000
           Nacho   j

In [7]:
# Aceptación de la cuarta pasada (auditada: 36/36 correctas, sin falsos positivos)
cols = ['player_id', 'tm_id', 'match_level', 'match_score']
n4_cons = n4[cols] if len(n4) else pd.DataFrame(columns=cols)

In [8]:
# Consolidación de los tres niveles de emparejamiento
cols = ['player_id', 'tm_id', 'match_level', 'match_score']
n3_cons = n3[cols] if len(n3) else pd.DataFrame(columns=cols)

matches_final = (pd.concat([matches[cols], n3_cons, n4_cons], ignore_index=True)
                 .drop_duplicates(subset='player_id', keep='first'))

n_total = len(wy)
print("COBERTURA FINAL DEL EMPAREJAMIENTO")
for lvl in ['exacto', 'difuso', 'difuso_recup', 'difuso_recup2']:    
    n = (matches_final['match_level'] == lvl).sum()
    print(f"  {lvl:<14}: {n:>5}  ({100*n/n_total:.1f}%)")
n_match = len(matches_final)
print(f"  {'TOTAL':<14}: {n_match:>5}  ({100*n_match/n_total:.1f}%)")
print(f"  {'sin valor':<14}: {n_total-n_match:>5}  ({100*(n_total-n_match)/n_total:.1f}%)")

COBERTURA FINAL DEL EMPAREJAMIENTO
  exacto        :  1584  (61.9%)
  difuso        :   766  (29.9%)
  difuso_recup  :    27  (1.1%)
  difuso_recup2 :    36  (1.4%)
  TOTAL         :  2413  (94.2%)
  sin valor     :   148  (5.8%)


### Cobertura final

El emparejamiento por cuatro niveles —exacto, difuso por similitud de tokens y recuperación de nombres usuales— enlaza 2.413 de los 2.561 jugadores del universo, una cobertura del 94,2%. Los 148 jugadores sin contraparte se reparten entre cinco ausencias reales en la fuente y un resto cuyos nombres divergen lo suficiente como para que ningún umbral los enlace con garantías. La inspección manual de las coincidencias difusas, ordenadas por puntuación ascendente, no reveló ningún falso positivo: el bloqueo sistemático por fecha de nacimiento hace estadísticamente improbable que dos jugadores distintos coincidan a la vez en fecha y en nombre similar. Se ha priorizado, por tanto, la fiabilidad del enlace sobre la cobertura máxima, decisión coherente con el carácter de descriptor de filtrado del valor de mercado: un enlace erróneo introduciría un precio falso en las recomendaciones, mientras que un jugador sin enlace simplemente no se filtra por coste y permanece disponible para el resto del sistema.

## 4. Alineación temporal de la valoración

El histórico de Transfermarkt asocia a cada jugador múltiples valoraciones fechadas a lo largo de su carrera, desde sus inicios hasta la actualidad. Tomar el valor más reciente sería un anacronismo: introduciría en un análisis de la temporada 2017/18 el precio que el jugador alcanzó años después, distorsionando por completo la lógica de scouting —un jugador que en 2017 era una promesa asequible podría figurar hoy con un valor de decenas de millones, o haberse retirado con valor nulo—.

Para cada jugador emparejado se selecciona, por tanto, la valoración contemporánea a la temporada analizada: la más reciente con fecha anterior o igual al cierre de la campaña 2017/18, fijado en el 30 de junio de 2018. Este criterio recupera el valor de mercado que el jugador tenía durante o justo antes de la temporada de la que proceden sus estadísticas de estilo, garantizando que precio y rendimiento se refieran al mismo momento. Los jugadores cuyo historial de valoraciones no alcanza esa fecha —altas posteriores en la base de datos— quedan sin valor asignado para la temporada, en coherencia con el criterio de no imputar datos ausentes.

In [9]:
# Selección de la valoración contemporánea a la temporada 2017/18
FECHA_CORTE = pd.Timestamp('2018-06-30')

tm_vals_w = tm_vals.copy()
tm_vals_w['date'] = pd.to_datetime(tm_vals_w['date'], errors='coerce')

# Solo valoraciones de jugadores emparejados y anteriores al cierre de temporada
ids_tm = set(matches_final['tm_id'])
vals = tm_vals_w[(tm_vals_w['player_id'].isin(ids_tm)) &
                 (tm_vals_w['date'] <= FECHA_CORTE)].copy()

# Para cada jugador, la valoración más reciente dentro de la ventana
vals = (vals.sort_values('date')
            .groupby('player_id', as_index=False)
            .last()[['player_id', 'date', 'market_value_in_eur']]
            .rename(columns={'player_id': 'tm_id',
                             'date': 'valuation_date',
                             'market_value_in_eur': 'market_value_eur'}))

con_valor = matches_final.merge(vals, on='tm_id', how='left')

n_match = len(con_valor)
n_con_valor = con_valor['market_value_eur'].notna().sum()
print("ALINEACIÓN TEMPORAL (corte 2018-06-30)")
print(f"  Emparejados              : {n_match}")
print(f"  · con valoración ≤ corte : {n_con_valor}  ({100*n_con_valor/n_match:.1f}%)")
print(f"  · sin valoración en ventana: {n_match-n_con_valor}")
print(f"\nRango de fechas de valoración usadas:")
print(f"  desde {con_valor['valuation_date'].min().date()} hasta {con_valor['valuation_date'].max().date()}")
print(f"\nDISTRIBUCIÓN DEL VALOR DE MERCADO (M€):")
mv = con_valor['market_value_eur'].dropna() / 1e6
print(mv.describe(percentiles=[.25, .5, .75, .95]).round(2).to_string())

ALINEACIÓN TEMPORAL (corte 2018-06-30)
  Emparejados              : 2413
  · con valoración ≤ corte : 2369  (98.2%)
  · sin valoración en ventana: 44

Rango de fechas de valoración usadas:
  desde 2012-08-13 hasta 2018-06-30

DISTRIBUCIÓN DEL VALOR DE MERCADO (M€):
count    2369.00
mean        8.56
std        14.60
min         0.05
25%         1.25
50%         3.50
75%         9.00
95%        35.00
max       180.00


## 5. Tabla final y persistencia

El resultado del enriquecimiento se consolida en una tabla de la capa Gold, `gold.player_market_values`, indexada por el identificador de jugador de Wyscout —el mismo `player_id` que emplea el resto del proyecto—, de modo que el notebook de clustering pueda incorporarla mediante una unión directa. Cada registro conserva, además del valor de mercado en euros, la fecha de la valoración utilizada, el identificador de Transfermarkt y el nivel de emparejamiento que originó el enlace, lo que mantiene la trazabilidad completa del proceso y permite auditar a posteriori el origen de cualquier valor.

La tabla cubre los 2.369 jugadores con valoración contemporánea a la temporada; el resto del universo permanece sin valor de mercado, en coherencia con el criterio de no imputar. Con esta tabla disponible, las funciones de scouting del notebook 05 podrán filtrar las recomendaciones de estilo por coste, completando la lógica de detección de oportunidades de mercado.

In [10]:
# Tabla gold.player_market_values: materialización y persistencia
gold_mv = con_valor[con_valor['market_value_eur'].notna()].copy()
gold_mv = gold_mv[['player_id', 'tm_id', 'market_value_eur',
                   'valuation_date', 'match_level', 'match_score']]
gold_mv['market_value_eur'] = gold_mv['market_value_eur'].astype('int64')
gold_mv['valuation_date'] = pd.to_datetime(gold_mv['valuation_date']).dt.date

gold_mv.to_sql('player_market_values', engine, schema='gold',
               if_exists='replace', index=False,
               dtype={'valuation_date': __import__('sqlalchemy').types.Date})

# Persistencia paralela en disco (coherente con artefactos del proyecto)
OUT = PROJECT_ROOT / "data" / "processed"
OUT.mkdir(parents=True, exist_ok=True)
gold_mv.to_parquet(OUT / "player_market_values.parquet", index=False)

# Verificación de lo escrito en BBDD
check = pd.read_sql("""
    SELECT COUNT(*) AS n, MIN(market_value_eur) AS min_eur,
           MAX(market_value_eur) AS max_eur, MAX(valuation_date) AS max_fecha
    FROM gold.player_market_values
""", engine)
print("TABLA gold.player_market_values ESCRITA")
print(check.to_string(index=False))
print(f"\nParquet: {(OUT / 'player_market_values.parquet').relative_to(PROJECT_ROOT)}")
print(f"\nReparto por nivel de emparejamiento:")
print(gold_mv['match_level'].value_counts().to_string())

TABLA gold.player_market_values ESCRITA
   n  min_eur   max_eur  max_fecha
2369    50000 180000000 2018-06-30

Parquet: data/processed/player_market_values.parquet

Reparto por nivel de emparejamiento:
match_level
exacto           1555
difuso            752
difuso_recup2      35
difuso_recup       27


In [11]:
# ¿De qué fechas son los primeros precios de los 41 que se quedaron fuera por el corte?
ids_sin_valor_ventana = set(matches_final['tm_id']) - set(tm_vals_w[tm_vals_w['date'] <= FECHA_CORTE]['player_id'])
primeros_post = (tm_vals_w[tm_vals_w['player_id'].isin(ids_sin_valor_ventana)]
                 .sort_values('date').groupby('player_id', as_index=False).first())
primeros_post['meses_tras_corte'] = ((primeros_post['date'] - FECHA_CORTE).dt.days / 30.44).round(1)
print(f"Primeros precios de los {len(primeros_post)} sin valor en ventana:\n")
print(primeros_post['date'].dt.to_period('M').value_counts().sort_index().to_string())
print(f"\nMeses tras el corte (distribución):")
print(primeros_post['meses_tras_corte'].describe(percentiles=[.25,.5,.75]).round(1).to_string())

Primeros precios de los 41 sin valor en ventana:

date
2018-07    4
2018-09    3
2018-10    2
2018-11    1
2018-12    1
2019-03    1
2019-04    2
2019-05    3
2019-06    5
2019-07    2
2019-08    1
2019-09    2
2019-10    4
2019-11    2
2019-12    1
2020-01    1
2020-08    4
2020-11    1
2022-07    1
Freq: M

Meses tras el corte (distribución):
count    41.0
mean     12.6
std       9.5
min       0.2
25%       5.7
50%      11.9
75%      15.4
max      48.9
